# TP3 - Convolutional Neural Networks

This TP is in two parts. In the first one, we will learn to create and train CNNs and ResNets using PyTorch. In the second one, we will look inside the neurons of VGG16, to see what patterns in images lead to large activations.

For this TP, you will need to fill in the blanks left in some cells of this notebook. They will be clearly indicated by comments of the form ```## TO FILL```. Also, because we will be training neural networks, I advise you to use an online Jupyter notebook environment that provides access to GPUs (e.g. Google Colab or the [SSPCloud](https://datalab.sspcloud.fr/)) if you don't have any on your laptop.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import torch
from torchvision import transforms, datasets, models
import torch.nn.functional as F
import torch.utils.data as data
import torch.nn as nn
import numpy as np
from tqdm import tqdm

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
device = torch.device(device)
print('Using device: %s ' % device)

### Helper functions

The next cell is just some helper functions for plotting, you do not need to have a look at them, and they are already used where needed in this notebook.

In [ ]:
def print_digits(images : torch.Tensor, labels : torch.Tensor):

    N = images.shape[0]

    fig, ax = plt.subplots(1, N, figsize=(N * 2, 2))
    for i in range(N):
        ax[i].imshow(images[i], cmap='gray')
        ax[i].set_title(f"Label: {labels[i].item()}")
        ax[i].axis('off')
    plt.show()


def print_image(image: torch.Tensor):

    fig, ax = plt.subplots(figsize=(2, 2))
    ax.imshow(image.squeeze(0).squeeze(0), cmap='gray')
    ax.axis('off')
    plt.show()


def plot_histories(histories):
    
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for color, (name, h) in zip(['tab:blue', 'tab:orange', 'tab:green', 'tab:red'], histories.items()):
        epochs = range(1, len(h["train_loss"]) + 1)
        axes[0].plot(epochs, h["train_loss"], '--', color=color, label=f"{name} (train)")
        axes[0].plot(epochs, h["val_loss"], '-', color=color, label=f"{name} (validation)")
        axes[1].plot(epochs, h["train_acc"], '--', color=color)
        axes[1].plot(epochs, h["val_acc"], '-', color=color)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].set_yscale("log"); axes[0].legend()
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy")
    plt.tight_layout(); plt.show()

## Part 1: Image Analysis with CNNs and ResNets in PyTorch

In this first part, the goal is to train a CNN to perform handwritten digit recognition, on the famous MNIST dataset. The MNIST dataset contains tens of thousands of $28 \times 28$ grayscale handwritten digits, with the corresponding labels in $\{0, \dots,9\}$. 

Trivia: This database was compiled in 1994, as part of the effort in the 1990s to standardize automation of sorting devices with human input, for instance sorting mail with handwritten postal codes at the post office. This is now often considered one of the first real successes of neural networks, and the first easy example on which performance of new such algorithms is tested.

### 1.a) Loading MNIST

The first cell below downloads the MNIST dataset in the ```/data/MNIST/``` directory, then converts it into PyTorch Datasets (which is the structure that lets you access your data samples), and finally converts it into Dataloaders (the structure which will be used to access batches of data during training and testing). In between, we use transforms, which are frequently used in image analysis. Here, the first transform converts the raw image to PyTorch tensors, and the second normalizes it (i.e. shifts and rescales it so that each coefficient has prescribed mean and standard deviation).

In [ ]:
root_dir = './data/MNIST/'

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root=root_dir, train=True, download=True, transform=transform)
val_set = datasets.MNIST(root=root_dir, train=True, download=True, transform=transform)
test_set = datasets.MNIST(root=root_dir, train=False, download=True, transform=transform)

train_idx = torch.arange(0, 50000)
val_idx = torch.arange(50000, 60000)

train_set.data, train_set.targets = train_set.data[train_idx], train_set.targets[train_idx]
val_set.data, val_set.targets = val_set.data[val_idx], val_set.targets[val_idx]

train_loader = data.DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = data.DataLoader(val_set, batch_size=500, shuffle=False)
test_loader = data.DataLoader(test_set, batch_size=500, shuffle=False)

We can see from the shape of the training dataset's data and targets that we have $60000$ training samples, with inputs of size $28 \times 28$. The last line gives all of the labels contained in the targets.

In [ ]:
images = train_set.data.to(torch.float32)
labels = train_set.targets
print(images.shape,labels.shape)
print(labels.unique())

We may also visualize some training samples, along with their labels.

In [ ]:
img_samples, label_samples = images[5000:5008], labels[5000:5008]
print_digits(img_samples, label_samples)

### 1.b) Convolutions and Pooling functions in PyTorch

Before creating and training CNNs, we are going to look at how convolutions and pooling work in PyTorch. We are going to test these functions on the following training sample. Here, we make the tensor 3D, because the functions we are going to use take as input tensors of 4 dimensions (batch_size x n_channels x height x width). The ```.unsqueeze(0).unsqueeze(0)``` adds a "batch" and a "channel" dimension to the image. 

In [ ]:
image = images[4].unsqueeze(0).unsqueeze(0)
print_image(image)
print(image.shape)

In the next cell, construct the following convolution filter as a PyTorch Tensor:
\begin{equation*}
    \begin{pmatrix}
        1 & 0 & -1 \\ 1 & 0 & -1 \\1 & 0 & -1 
    \end{pmatrix}
\end{equation*}
Keep in mind that filters should also be 4D tensors, of size (batch_size x n_channels x filter_height x filter_width).

In [ ]:
filter = ## TO FILL

print(filter.shape) # Should be (1, 1, 3, 3)

Next, use the ```F.conv2d``` function from PyTorch to apply the filter to the image. Before running the cell, try to predict the size of the output. What if you add padding, or have a stride $>1$? 

In [ ]:
conv_image = ## TO FILL

print_image(conv_image)
print("Output shape:", conv_image.shape)

What kind of patterns does the filter we defined above detect? Redo the previous cells with a filter that will detect horizontal edges, and next with a larger filter (e.g. of size 5 x 5).

Next, test the average pooling and max pooling functions on this same image, let's say with stride 2 and kernel size 3. They are called ```F.avg_pool2d``` and ```F.max_pool2d```. Once again, try to predict the size of the output before running the cells, and try it for different values of stride and kernel_size.

In [ ]:
avgpool_image = ## TO FILL

print_image(avgpool_image)
print("Output shape:", avgpool_image.shape)

In [ ]:
maxpool_image = ## TO FILL

print_image(maxpool_image)
print("Output shape:", maxpool_image.shape)

### 1.c) Training a CNN

In the previous section, we applied convolution and pooling *functions* (```F.conv2d```, ```F.max_pool2d```) with filters that we designed by hand. In this section, we will build a CNN with the corresponding PyTorch *modules* (```nn.Conv2d```, ```nn.MaxPool2d```). The difference is that the filters are now **parameters of the network**, learned by gradient descent so as to minimize the classification loss, instead of being hand-crafted.

We are going to build the following CNN. Padding and stride will be kept to the default values (0 and 1 respectively).

| Layer | Output channels | Kernel size |
|---|---|---|
| Convolution + ReLU | $8$ | $3 \times 3$ |
| Max pooling | | $2 \times 2$ |
| Convolution + ReLU | $16$ | $3 \times 3$ |
| Max pooling | | $2 \times 2$ |
| Flatten + Linear | $10$ (one per digit) | |

Fill in the cell below to create the CNN, which will be a ```nn.Module```. In a ```nn.Module```, we define in ```__init__``` the layers of the CNN as attributes (here, you will need to define ```self.conv1```, ```self.relu``` and so on ...). ```forward``` is the function which performs a forward pass, and returns the output of the CNN when given `x` as an input. It makes use of the layers defined in ```__init__```.

The layers you are going to use are ```nn.Conv2d```, ```nn.ReLU```, ```nn.MaxPool2d```, ```nn.Flatten``` and ```nn.Linear```. ```nn.Flatten``` turns 4d tensors of the form (batch_size, n_channels, height, width) into 2D tensors of the form (batch_size, n_channels * height * width). This is necessary because ```nn.Linear``` applies only to the last dimension of the tensor given as an input.

NB: you will need to compute by hand the size of the output of the last Max pooling layer, to find the number of in_features the last linear layer must take.

In [ ]:
class CNN(torch.nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        ## TO FILL : define the layers of the CNN
        self.conv1 = 
        self.conv2 = 
        self.linear = 
        self.relu = 
        self.maxpool = 
        self.flatten = 
       
    def forward(self, x):
        ## TO FILL: Apply the layers to the input x using the attributes in __init__, and return the output.
        return 

cnn = CNN()

The output of ```cnn``` should be a tensor of shape (batch_size, n_classes). It is generally good practice to test that everything is alright with the sizes using a dummy input before starting training.

In [ ]:
# Testing on a fake input image of size (batch_size, n_channels, height, width)
x = torch.randn(1, 1, 28, 28)   
print(f"Output shape: {cnn(x).shape}") # Should be of size (batch_size, 10)

As a baseline, we will compare our newly defined CNN with a simple fully-connected MLP, defined as follows. Note the use of the convenient ```nn.Sequential``` function, which allows you to directly create a ```nn.Module``` by stacking layers given as argument.

In [ ]:
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128), 
    nn.ReLU(),
    nn.Linear(128, 64), 
    nn.ReLU(),
    nn.Linear(64, 10),
)

We can compare the number of parameters between the CNN and MLP as follows.

In [ ]:
cnn_params = sum(p.numel() for p in cnn.parameters())
mlp_params = sum(p.numel() for p in mlp.parameters())
print(f"Number of parameters in CNN: {cnn_params}")
print(f"Number of parameters in MLP: {mlp_params}")

Now we need to train these models. The training of a model on one epoch (= one pass over the whole training set) always follows the same recipe, for each batch of images:
1. set the gradients to zero,
2. compute the output of the model (*forward pass*),
3. compute the loss between the output and the true labels,
4. compute the gradients of the loss with respect to the parameters (*backward pass*),
5. update the parameters with the optimizer (here, Adam).

We use ```model.train()``` and ```model.eval()``` to tell the model whether it is being trained or evaluated. During evaluation, we use ```torch.no_grad()``` since we do not need gradients.

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, n_correct = 0., 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(targets)
        n_correct += (outputs.argmax(dim=1) == targets).sum().item()
    return total_loss / len(loader.dataset), n_correct / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, n_correct = 0., 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        total_loss += criterion(outputs, targets).item() * len(targets)
        n_correct += (outputs.argmax(dim=1) == targets).sum().item()
    return total_loss / len(loader.dataset), n_correct / len(loader.dataset)


def fit(model, n_epochs=5, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(n_epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer)
        val_loss, val_acc = evaluate(model, val_loader)
        for key, value in zip(history, [train_loss, train_acc, val_loss, val_acc]):
            history[key].append(value)
        print(f"Epoch {epoch + 1:02d} | train loss {train_loss:.4f}, acc {100 * train_acc:.2f}% "
              f"| val loss {val_loss:.4f}, acc {100 * val_acc:.2f}%")
    return history

Let's train the CNN and the MLP for a few epochs. This should take a couple of minutes at most. Compare the training and validation curves of the two models. The CNN should beat the MLP, even though it has fewer parameters.

In [ ]:
n_epochs = 5

print("Training the CNN")
history_cnn = fit(cnn, n_epochs)

print("\nTraining the MLP")
history_mlp = fit(mlp, n_epochs)

plot_histories({"CNN": history_cnn, "MLP": history_mlp})

print(f"Test accuracy of the CNN: {100 * evaluate(cnn, test_loader)[1]:.2f}%")
print(f"Test accuracy of the MLP: {100 * evaluate(mlp, test_loader)[1]:.2f}%")

### 1.d) Training a ResNet

Convolutional networks become more powerful when they are deeper, but they are also harder to train. In this section, we will see why, and how **residual connections** solve this problem. Recall that a residual block computes
$$x^{(l+1)} = x^{(l)} + g_\theta(x^{(l)}),$$
where $g_\theta$ is a small network (here, two convolutions of size $3\times 3$). The input is added to the output of the block: this is called a *skip connection*. For this addition to be possible, $g_\theta(x)$ must have the **same shape** as $x$: we use $3 \times 3$ convolutions with ```padding=1``` and the same number of input and output channels.

Fill in the ```forward``` method of the following block: $g_\theta$ is a convolution, followed by a ReLU, followed by a second convolution. If ```residual=True```, add the input $x$ to the output. Then, apply a final ReLU. The ```residual``` argument will let us compare with the same block *without* the skip connection.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels, residual = True):
        super().__init__()
        self.residual = residual
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        ## TO FILL
        return

block = ResidualBlock(channels=8)
print(block(torch.randn(4, 8, 14, 14)).shape)  # The shape should be unchanged: (4, 8, 14, 14)

We now stack these blocks to build a deep network. It starts with a convolution (from $1$ to ```channels``` channels), a ReLU and a max pooling, which reduces the image to $14 \times 14$, followed by ```n_blocks``` blocks. It ends with a **global average pooling** (which averages each channel over all the positions of the image, giving one value per channel) and a linear layer. Fill in the cell to stack the blocks with ```nn.Sequential```. Make sure to use the ```residual``` argument when using ```ResidualBlock```.

Hint: Use the unpacking operator ```*```, which can be put before a list of arguments ```arg = [arg1, arg2, ...]``` so that calling a function ```f(*arg)``` is the same as calling ```f(arg1, arg2, ...)```.

In [ ]:
class DeepCNN(nn.Module):
    def __init__(self, n_blocks, channels=8, residual = True):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, channels, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.blocks = ## TO FILL: stack n_blocks residual blocks
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(channels, 10))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))

model = DeepCNN(n_blocks=10)
print(model(torch.randn(4, 1, 28, 28)).shape)

The problem of very deep networks without residual connections is that the gradients **vanish**: when backpropagating through many layers, the gradient is multiplied by the Jacobian of each layer, and its norm gets smaller and smaller. The first layers of the network then barely learn.

With a skip connection, the Jacobian (derivative) of a block is $I + \frac{\partial g_\theta}{\partial x}$: the identity term lets the gradient flow directly through the network.

Let's check this. For networks with a varying number of blocks, with and without skip connections, we compute the gradient of the loss (on one batch) with respect to the weights of the **first** convolution, and look at its norm. Before running the cell, what do you expect to see as the depth increases, in each case?

In [ ]:
def first_layer_grad_norm(model):
    model.to(device).train()
    inputs, targets = next(iter(train_loader))
    loss = criterion(model(inputs.to(device)), targets.to(device))
    model.zero_grad()
    loss.backward()
    return model.stem[0].weight.grad.norm().item()

depths = [1, 2, 5, 10, 20]
torch.manual_seed(0)
grad_plain = [first_layer_grad_norm(DeepCNN(d, residual=False)) for d in depths]
grad_resnet = [first_layer_grad_norm(DeepCNN(d, residual=True)) for d in depths]

plt.figure(figsize=(5, 3.5))
plt.plot(depths, grad_plain, 'o-', label="without skip connections")
plt.plot(depths, grad_resnet, 'o-', label="with skip connections (ResNet)")
plt.yscale("log"); plt.xlabel("number of blocks"); plt.ylabel("gradient norm (first layer)"); plt.legend(); plt.show()

Let's now train both networks with $10$ blocks (i.e. $21$ convolutional layers). This can take a few minutes.

In [ ]:
n_epochs = 3

torch.manual_seed(0)
print("Training the network without skip connections")
plain_net = DeepCNN(n_blocks=10, residual=False)
history_plain = fit(plain_net, n_epochs)

torch.manual_seed(0)
print("\nTraining the ResNet")
resnet = DeepCNN(n_blocks=10, residual=True)
history_resnet = fit(resnet, n_epochs)

plot_histories({"No skip": history_plain, "ResNet": history_resnet})

print(f"Test accuracy of the network without skip connections: {100 * evaluate(plain_net, test_loader)[1]:.2f}%")
print(f"Test accuracy of the ResNet: {100 * evaluate(resnet, test_loader)[1]:.2f}%")

Compare the two curves. Does the network without skip connections manage to learn? Compare the final accuracy of the ResNet with the one of the simple CNN of the previous section: is it better? Note that this is a very simple task (MNIST), on which the benefits of deep networks are limited, but the training behavior would be similar on more difficult tasks.

## Part 2: Visualizing the learned features in VGG16

In this part, we are going to try to visualize the features learned by VGG16 (a large CNN). We first load VGG16.

In [ ]:
model_vgg = models.vgg16(weights='DEFAULT')
model_vgg.to(device)
print(model_vgg)

We can visualize the convolution filters of VGG16, e.g. of the first layer:

In [ ]:
filters = model_vgg.features[0].weight

fig, axs = plt.subplots(8, 8, figsize=(15, 15))
for i, f in enumerate(filters[:,0,:,:]):
    axs[i % 8, int(i / 8)].imshow(f.cpu().detach().numpy())
    axs[i % 8, int(i / 8)].set_title(i)

plt.show()

In the module `ChannelActivation(layer, channel)`, fill in the ```forward``` method that returns the average activation (i.e. output value) of channel `channel` of layer `layer` of the VGG features.

In [ ]:
class ChannelActivation(nn.Module):
    def __init__(self, layer, channel):
        super().__init__()
        self.layers = nn.Sequential(model_vgg.features[:layer+1])
        self.channel = channel
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x):
        ## TO FILL
        return

Our objective is to find which patterns are recognized by a given channel. To do so, we will follow the approach of [this Distill article](https://distill.pub/2017/feature-visualization/) and find images that lead to the highest possible channel activation.

First, create a random (colored) image of size 128x128, initialized with values drawn at random between 0.4 and 0.6 (i.e. grey + small perturbation). Then, perform 200 steps of Adam (with lr=0.01) to maximize the activation of channel 4 of layer 1. Plot the image after 0, 10, 50, 100 and 200 iterations. You should see a pink saturated image with several horizontal lines, indicating that the channel probably recognizes horizontal edges.

**NB1:** Careful, by default, optimizers minimize their objective, not maximize it!

**NB2:** The parameters given to an optimizer should be on the cpu. If you use a gpu, you thus need to keep two versions of the image: 1) a cpu version given to the optimizer, and 2) a gpu version, created at each iteration of the optimization, and used to compute the gradient.

In [ ]:
tensor_to_image = transforms.ToPILImage()  # Only used for plotting

input_image = ## TO FILL
input_image.requires_grad = True
activation = ## TO FILL
activation.to(device)
optimizer = ## TO FILL

fig, axs = plt.subplots(1, 5, figsize=(20, 20))
i_plot = 0
for epoch in tqdm(range(201)):
    if epoch in [0,10,50,100,200]:
        axs[i_plot].imshow(tensor_to_image(input_image))
        axs[i_plot].axis('off')
        i_plot += 1

    ## TO FILL : Compute the activation and update the image to MAXIMIZE it (hint: use the optimizer, backward(), step(), don't forget about zero_grad())

plt.show()

Let's check the highest and lowest values of the image.

In [ ]:
print(torch.min(input_image).item(), torch.max(input_image).item())

This is an issue, because VGG16 takes as inputs images with values in $[0,1]$. To avoid this problem, clip the image pixels to $[0.2,0.8]$ after each optimization step using `input_image.data = input_image.data.clip(0.2, 0.8)`. You should now see several clear horizontal lines in a blue background.

In [ ]:
tensor_to_image = transforms.ToPILImage()

input_image = ## TO FILL
input_image.requires_grad = True
activation = ## TO FILL
activation.to(device)
optimizer = ## TO FILL

fig, axs = plt.subplots(1, 5, figsize=(20, 20))
i_plot = 0
for epoch in tqdm(range(201)):
    if epoch in [0,10,50,100,200]:
        axs[i_plot].imshow(tensor_to_image(input_image))
        axs[i_plot].axis('off')
        i_plot += 1

    ## TO FILL : same as before, but add a line to clip the image values between 0.2 and 0.8 after the optimizer step
    
plt.show()

One issue with our current code is that VGG was trained on **normalized** images, and thus is not adapted to our input image. To normalize the image, we will use **transforms**.

Create a function `create_activation_image(layer, channel, transform=None, image_size=128, show_steps=False)` that maximizes the corresponding channel activation on an image of size `image_size`, and first applies `transform` to the image before computing the gradient of the activation. 

Then, test your function with `transform=transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])`. Is this better? You should now see a horizontal pattern with lines.

In [ ]:
## TO FILL
def create_activation_image(layer, channel, transform=None, image_size=128, show_steps=False):
    tensor_to_image = transforms.ToPILImage()

    input_image = ## TO FILL
    input_image.requires_grad = True
    activation = ## TO FILL
    activation.to(device)
    optimizer = ## TO FILL

    if show_steps:
        fig, axs = plt.subplots(1, 5, figsize=(20, 20))
        i_plot = 0
        for ax in axs:
            ax.axis('off')

    for epoch in tqdm(range(201)):
        if show_steps and epoch in [0,10,50,100,200]:
            axs[i_plot].imshow(tensor_to_image(input_image))
            i_plot += 1

        ## TO FILL : same as before, but transform the image before computing the activation, and clip the image values between 0.2 and 0.8 after the optimizer step

    if show_steps:
        plt.show()

    return input_image

image = create_activation_image(1, 4, 
                                transform=transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), 
                                show_steps=True)

Now test your function on channel 0 of layer 20. The pattern that appears should vaguely resemble fish scales.

In [ ]:
image = ## TO FILL

Large neural networks are prone to adversarial attacks, i.e. a small well-crafted additive noise can dramatically change the output of the model, and thus lead to incorrect classification. For our purpose, this is an issue, as the optimization algorithm may find such very specific noise instead of more valuable visual patterns.

To avoid this issue and further improve our images, we are thus going to apply small random perturbations to the image before computing the gradient. This will prevent the optimizer from optimizing the noise, and overall increase the stability of our process.

To do so, add a composition of several transforms using `transforms.Compose`:

1.   A small pixel noise with `transforms.Lambda(lambda x: x + 0.001 * (2 * torch.rand_like(x) - 1))`
2.   A random affine transform with `transforms.RandomAffine(degrees=5, translate=(0.1,0.1), scale=(0.9,1.1))`
3.   A random crop of size 96 using `transforms.RandomCrop`
4.   Random local fluctations with `transforms.ElasticTransform(alpha=50.)`
5. The same normalization as previously

Compare the activation images with and without these random transformations. Is the pattern more visible?



In [ ]:
random_transforms = ## TO FILL

image = create_activation_image(20, 0, transform=transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), show_steps=True)
image = create_activation_image(20, 0, transform=random_transforms, show_steps=True)

To see what the transformation is doing to the image, we apply the random transformations (without normalization) to the following simple image, and show 5 randomly transformed images.

In [ ]:
sample_image = 0.3 * torch.ones(3, 256, 256)
sample_image[0,:,40:80] += 0.7
sample_image[1,10:20,:] += 0.5
sample_image[2,150:,:] += 0.5
plt.imshow(tensor_to_image(sample_image))

random_transforms_withoutnorm = ## TO FILL

fig, axs = plt.subplots(1, 5, figsize=(20, 20))
i_plot = 0
for i_plot in range(5):
    perturbed_image = random_transforms_withoutnorm(sample_image)
    axs[i_plot].imshow(tensor_to_image(perturbed_image))
plt.show()

Finally, show the activation images for the first 5 channels of layers [1, 10, 20, 30]. You should be able to see a gradual complexification of the patterns.

**PS1:** Our method seems unable to find meaningful patterns for the last layer. One issue is probably that the random crop imposes that all regions on the image look similar (as they all should have a high channel activation), thus preventing larger and more complex patterns to emerge from the optimization.

**PS2:** You can also try other layers and channels to find interesting patterns!

In [ ]:
## TO FILL